# gVCF to VDS

In [1]:
%%configure -f
{
    "driverMemory": "45G"
}

In [2]:
# Import and initiate HAIL
import hail as hl
hl.init(sc)

Starting Spark application


ID,YARN Application ID,Kind,State,Spark UI,Driver log,User,Current session?
0,application_1745888546191_0001,pyspark,idle,Link,Link,None,✔


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

SparkSession available as 'spark'.


FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

pip-installed Hail requires additional configuration options in Spark referring
  to the path to the Hail Python module directory HAIL_DIR,
  e.g. /path/to/python/site-packages/hail:
    spark.jars=HAIL_DIR/backend/hail-all-spark.jar
    spark.driver.extraClassPath=HAIL_DIR/backend/hail-all-spark.jar
    spark.executor.extraClassPath=./hail-all-spark.jarRunning on Apache Spark version 3.5.2-amzn-1
SparkUI available at http://ip-192-168-71-146.ap-southeast-1.compute.internal:38371
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.134-952ae203dbbe
LOGGING: writing to /mnt/yarn/usercache/livy/appcache/application_1745888546191_0001/container_1745888546191_0001_01_000001/hail-20250429-0114-0.2.134-952ae203dbbe.log

## Load 1KG gVCF into VDS

- List the single sample hard-filtered.gvcf.gz generated for 1000 genomes dragen 3.7.6 analysis
- Upload the list (csv file) into S3
- Set gvcf_list_path

In [3]:
gvcf_list_path='s3://npm-grids/hebrardms/batch/sg10k_reprocess_gvcf_manifest.csv'
vds_prefix = 's3://precise-scratch/hebrardms/SG10K_Health/VDS'

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [4]:
# Import csv samples to ht
ht_gvcf = hl.import_table(gvcf_list_path, delimiter=',', quote = '"', no_header=True)
# Rename the columns
ht_gvcf = ht_gvcf.rename({'f0': 'bucket', 'f1': 'prefix'})
# Build S3 path
ht_gvcf = ht_gvcf.annotate(
    s3_path = hl.str('s3://') + hl.str(ht_gvcf.bucket) + '/' + hl.str(ht_gvcf.prefix)
)

ht_gvcf.count()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

10322
2025-04-29 01:14:35.728 Hail: INFO: Reading table without type imputation
  Loading field 'f0' as type str (not specified)
  Loading field 'f1' as type str (not specified)

In [5]:
# List of S3 path
ls_gvcf = ht_gvcf.s3_path.collect()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

In [ ]:
## Test
###
# gvcf_paths=ls_gvcf[0:10] # variant_data: 13905139 rows and 10 columns in 2586 partitions ~ 30Gb ~ 2h on 9 CPU onDemand
# gvcf_paths=ls_gvcf[0:100] # variant_data: 37755085 rows and 100 columns in 2586 partitions ~ 30Gb ~ 30min on 500 CPU onDemand

In [8]:
# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-bf50-tr100k-sp20k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    gvcf_paths=ls_gvcf[0:1000],
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=50, # number of inputs combined in one VDS
    target_records=100000 # number of rows per partition
)

combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
ConnectionPoolTimeoutException: Timeout waiting for connection from pool

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 2159 in stage 28.0 failed 4 times, most recent failure: Lost task 2159.3 in stage 28.0 (TID 124567) (ip-192-168-67-67.ap-southeast-1.compute.internal executor 373): com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.SdkClientException: Unable to execute HTTP request: Timeout waiting for connection from pool
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleRetryableException(AmazonHttpClient.java:1219)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeHelper(AmazonHttpClient.java:1165)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.doExecute(AmazonHttpClient.java:814)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.execute

In [9]:
# Combine VDS

# List of VDS
# 1,000 samples -> 20 VDS of 50 samples each
ls_vds = [
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_00.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_01.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_02.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_03.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_04.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_05.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_06.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_07.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_08.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_09.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_10.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_11.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_12.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_13.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_14.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_15.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_16.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_17.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_18.vds",
    "s3://precise-scratch/hebrardms/SG10K_Health/VDS/checkpoints/combiner-intermediates/d57251e4-b3ce-4360-a1d5-fae646837a33_gvcf-combine_job1/dataset_19.vds",
]

# Change the number of tasks hail is launching in parallel
hl._set_flags(spark_max_stage_parallelism='20000')

# Combine gVCF
combiner = hl.vds.new_combiner(
    output_path=f'{vds_prefix}/SG10K_Health_1k-bf10-tr100k-sp20k.vds',
    temp_path=f'{vds_prefix}/checkpoints/',
    vds_paths=ls_vds,
    use_genome_default_intervals=True,
    reference_genome='GRCh38',
    branch_factor=10, # number of inputs combined in one VDS
    target_records=100000 # number of rows per partition
)
combiner.run()

FloatProgress(value=0.0, bar_style='info', description='Progress:', layout=Layout(height='25px', width='50%'),…

An error was encountered:
ConnectionPoolTimeoutException: Timeout waiting for connection from pool

Java stack trace:
org.apache.spark.SparkException: Job aborted due to stage failure: Task 6054 in stage 35.0 failed 4 times, most recent failure: Lost task 6054.3 in stage 35.0 (TID 146713) (ip-192-168-74-174.ap-southeast-1.compute.internal executor 456): com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.SdkClientException: Unable to execute HTTP request: Timeout waiting for connection from pool
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.handleRetryableException(AmazonHttpClient.java:1219)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.executeHelper(AmazonHttpClient.java:1165)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.doExecute(AmazonHttpClient.java:814)
	at com.amazon.ws.emr.hadoop.fs.shaded.com.amazonaws.http.AmazonHttpClient$RequestExecutor.execut